# Fitting the cosmological constant to supernovae

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/cosmological_constant_fit)

1466 real Type Ia supernovae from the open Pantheon+ compilation are fitted with a Friedmann universe of matter $\Omega_m$, cosmological constant $\Omega_\Lambda$ and curvature. One batched $\chi^2$ evaluates thousands of universes at every supernova at once and draws the $\Omega_m$–$\Omega_\Lambda$ confidence contours, while Adam and autograd walk from the matter-only universe to the best fit. Distant supernovae are fainter than in a decelerating universe: $q_0 < 0$, the expansion is accelerating.

**Vector form**

$$E(z)^2 = \Omega_m(1+z)^3 + \Omega_k(1+z)^2 + \Omega_\Lambda,\qquad d_L = (1+z)\,\frac{c}{H_0}\,S_k\!\Big(\int_0^z\frac{dz'}{E(z')}\Big),\qquad \mu = 5\log_{10}\frac{d_L}{\mathrm{Mpc}} + 25,\qquad q_0 = \frac{\Omega_m}{2} - \Omega_\Lambda$$

**Index (tensor) form**

$$\chi_{abk} = \sum_{k'<k}\tfrac{h}{2}\big(E^{-1}_{abk'} + E^{-1}_{ab,k'+1}\big),\qquad \mu_{abi} = 5\log_{10}\!\big[(1+z_i)\,S_k(\chi_{ab}(z_i))\big] + 25 - 5\log_{10}H_0,\qquad \chi^2_{ab} = w_i\,(\mu_i - \mu_{abi})(\mu_i - \mu_{abi})$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $z_i,\ \mu_i,\ \sigma_i$ | `load_distance_moduli(path)` | (N,) | redshift, distance modulus and error of supernova $i$, read from any CSV with those columns |
| $\Omega_{m,a},\ \Omega_{\Lambda,b}$ | `parameter_grid(A, B, ...)` | (A,), (B,) | candidate universes; broadcast as $(A,1,1)$ and $(1,B,1)$ |
| $E^2_{abk}$ | `e_squared(nodes, Om, OL)` | (A, B, Z) | expansion rate at each node $z_k$ for every universe at once |
| $\chi_{abk}$ | `comoving_integral(nodes, Om, OL)` | (A, B, Z) | cumulative trapezoid of $1/E$ along the last axis |
| $\chi_{ab}(z_i)$ | `interpolate(nodes, chi, f, z)` | (A, B, N) | cubic Hermite interpolation to the supernova redshifts (slope $1/E$ known) |
| $S_k(\chi)$ | `transverse(chi, Ok)` | (A, B, N) | $\sinh$, identity or $\sin$ for open, flat or closed space |
| $-5\log_{10}H_0$ | `einsum('abi,i->ab', r, w) / w.sum()` | (A, B) | $i$ summed: the best $H_0$ of each universe is an analytic weighted mean |
| $\chi^2_{ab}$ | `einsum('abi,abi,i->ab', r, r, w)` | (A, B) | $i$ summed with weights $w_i = 1/\sigma_i^2$ |
| $\partial\chi^2/\partial(H_0, \Omega_m, \Omega_\Lambda)$ | `loss.backward()` | (3,) | autograd through the integral, used by Adam and L-BFGS |
| $C = 2\,H^{-1}$ | `autograd.functional.hessian(chi2, p)` | (3, 3) | covariance of the fitted parameters from the curvature of $\chi^2$ |

In 1998 two teams, the Supernova Cosmology Project (Perlmutter et al.) and the High-z Supernova Search Team (Riess et al.; Schmidt), found that distant Type Ia supernovae, which are standardisable candles, look fainter than a decelerating universe allows: the expansion is accelerating, which in general relativity requires something like a cosmological constant $\Lambda$ (dark energy). This won the 2011 Nobel Prize in Physics. Here the same test runs on today's data. The left panel plots each distance modulus minus that of an empty, coasting universe ($\Omega_m = \Omega_\Lambda = 0$): a matter-only universe (red) would make distant supernovae brighter (below zero), the data sit above. The right panel shows the 1, 2 and 3$\sigma$ regions ($\Delta\chi^2 = 2.30, 6.18, 11.83$) from the batched grid, with the flat line, the line $q_0 = 0$ separating acceleration from deceleration, and the "no Big Bang" corner where $E^2$ turns negative within the data. With these diagonal errors the fit gives $\Omega_m \approx 0.30 \pm 0.07$, $\Omega_\Lambda \approx 0.57 \pm 0.10$ (flat: $\Omega_m \approx 0.35 \pm 0.02$), and $\Omega_\Lambda \le 0$ is rejected at about $5.5\sigma$ ($\sigma = \sqrt{\Delta\chi^2}$ against the best model on the rejected side). $H_0$ is degenerate with the supernova absolute magnitude; the value printed ($\approx 73$ km/s/Mpc) comes from the SH0ES Cepheid calibration built into these $\mu$ and is not an independent measurement (the "Hubble tension" with the CMB value $\approx 67$ is an open question). Data: Pantheon+ distance moduli (Scolnic et al. 2022, ApJ 938, 113; Brout et al. 2022, ApJ 938, 110), from github.com/PantheonPlusSH0ES/DataRelease (file Pantheon+SH0ES.dat; publicly released for research use without a stated licence, the papers are open access under CC BY 4.0; please cite them). Kept: zHD, MU_SH0ES and MU_SH0ES_ERR_DIAG for $z > 0.01$, Cepheid-calibrator hosts removed, one row per supernova. Your own table works too: any CSV with columns z, mu, sigma_mu (see load_distance_moduli). Simplifications: only the diagonal errors are used, not the full statistical + systematic covariance of Pantheon+ ($\chi^2/\mathrm{dof} \approx 0.44$ shows these errors are conservative, while ignoring correlated systematics makes the contours somewhat optimistic); dark energy is a pure constant ($w = -1$); radiation is neglected.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/cosmological_constant_fit/main.py
"""Main loop: supernova table → batched χ² grid of universes → Adam fit → significance of acceleration."""

from pathlib import Path

from problems._common.runtime import cli, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # CPU and float64 throughout: the table is small and χ² differences of a few units matter.
    fast = fast_mode()
    z, mu, sigma = tensors.load_distance_moduli(tensors.DATA, z_min=params["z_min"])
    print(f"{len(z)} Type Ia supernovae from {tensors.DATA.name}, z = {z.min():.3f} … {z.max():.2f}")

    n = 31 if fast else params["grid"]
    Om, OL = tensors.parameter_grid(n, n, (0.0, 1.2), (-0.4, 1.6))
    chi2, H0_grid = solve.chi2_grid(z, mu, sigma, Om, OL)
    sig = solve.significance(chi2, Om, OL)
    print(
        f"χ² grid of {n}×{n} universes (H0 solved analytically in each): min χ² = {sig['chi2_min']:.1f} for "
        f"{len(z)} SNe at Ω_m = {sig['best_Om']:.2f}, Ω_Λ = {sig['best_OL']:.2f}, q0 = {sig['best_q0']:+.2f}"
    )
    print(
        f"Ω_Λ ≤ 0 rejected at {sig['sigma_lambda']:.1f}σ, a decelerating universe (q0 ≥ 0) at {sig['sigma_accel']:.1f}σ"
    )
    print(f"share of the likelihood with q0 < 0 (accelerating): {sig['p_accel']:.8f}")

    flat = params["geometry"] == "flat"
    steps = 40 if fast else params["steps"]
    fit = solve.fit(z, mu, sigma, (70.0, params["start_Om"], params["start_OL"]), steps, params["lr"], flat)
    (H0, om, ol), err = fit["best"].tolist(), fit["errors"].tolist()
    print(
        f"Adam ({steps} steps) + L-BFGS, {'flat' if flat else 'free curvature'}: "
        f"χ² {fit['chi2'][0]:.0f} → {fit['chi2'][-1]:.1f}"
    )
    print(f"H0 = {H0:.2f} ± {err[0]:.2f} km/s/Mpc   (set by the SH0ES Cepheid calibration of μ, not by the SNe alone)")
    print(f"Ω_m = {om:.3f} ± {err[1]:.3f}   Ω_Λ = {ol:.3f} ± {err[2]:.3f}   q0 = {equation.deceleration(om, ol):+.3f}")
    print(
        f"χ²/dof = {fit['chi2'][-1]:.0f}/{len(z) - (2 if flat else 3)}: below 1, the diagonal errors are conservative"
    )
    return plot.animate(z, mu, sigma, fit, Om, OL, chi2, sig, out_dir / "cosmological_constant_fit.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/cosmological_constant_fit/equation.py
r"""Physics → tensor translation: distances in a Friedmann–Lemaître universe.

The expansion rate of a universe with matter Ω_m, a cosmological constant Ω_Λ and curvature Ω_k = 1 − Ω_m − Ω_Λ is

    H(z) = H_0\,E(z),        E(z)^2 = Ω_m (1+z)^3 + Ω_k (1+z)^2 + Ω_Λ.

Light from a supernova at redshift z has crossed the (dimensionless) comoving distance

    χ(z) = \int_0^z dz' / E(z'),

and its flux falls off with the luminosity distance

    d_L = (1+z)\,\frac{c}{H_0}\,S_k(χ),   S_k(χ) = \sinh(\sqrt{Ω_k} χ)/\sqrt{Ω_k}   (open, Ω_k > 0)
                                           = χ                                      (flat)
                                           = \sin(\sqrt{-Ω_k} χ)/\sqrt{-Ω_k}       (closed, Ω_k < 0).

Astronomers quote the distance modulus μ = m − M = 5 \log_{10}(d_L / \mathrm{Mpc}) + 25.

Every function broadcasts: the cosmological parameters can carry leading batch axes (a whole grid of models, shape
(A, B, 1)) while the redshifts sit on the last axis, so one call evaluates every model at every supernova.
"""

import torch

C_KMS = 299792.458  # speed of light in km/s, so c/H0 is in Mpc when H0 is in km/s/Mpc


def e_squared(z: torch.Tensor, Om: torch.Tensor, OL: torch.Tensor) -> torch.Tensor:
    r"""E(z)^2 = Ω_m (1+z)^3 + Ω_k (1+z)^2 + Ω_Λ.   z: (Z,), Om, OL: (..., 1) → (..., Z)"""
    Ok = 1 - Om - OL
    x = 1 + z
    return Om * x**3 + Ok * x**2 + OL


def comoving_integral(nodes: torch.Tensor, Om: torch.Tensor, OL: torch.Tensor):
    r"""χ at every node: cumulative trapezoid of 1/E on the uniform grid z_0 = 0 < z_1 < … (Z,) → (..., Z).

    Also returns the integrand f = dχ/dz = 1/E at the nodes, used by `interpolate`. E² is clamped at a tiny
    positive value so the models with no Big Bang inside the data range (E² < 0 somewhere, see `valid`) stay
    finite; they are excluded from the fit by `valid`, not here.
    """
    f = e_squared(nodes, Om, OL).clamp_min(1e-12).rsqrt()  # 1/E, (..., Z)
    h = nodes[1] - nodes[0]
    steps = 0.5 * h * (f[..., 1:] + f[..., :-1])
    return torch.cat([torch.zeros_like(f[..., :1]), steps.cumsum(-1)], dim=-1), f


def interpolate(nodes: torch.Tensor, chi_nodes: torch.Tensor, f_nodes: torch.Tensor, z: torch.Tensor):
    """Cubic Hermite interpolation of χ from the grid to the supernova redshifts.   (..., Z) → (..., N)

    Hermite uses the value χ and the slope f = 1/E at both neighbouring nodes. The indices and weights depend
    only on the redshifts, so the result stays differentiable in Ω.
    """
    h = nodes[1] - nodes[0]
    k = (z / h).floor().long().clamp(0, len(nodes) - 2)  # (N,) left node of each supernova
    s = z / h - k  # (N,) position between the nodes, in [0, 1)
    h00, h10, h01, h11 = 2 * s**3 - 3 * s**2 + 1, s**3 - 2 * s**2 + s, 3 * s**2 - 2 * s**3, s**3 - s**2
    return (
        h00 * chi_nodes[..., k]
        + h10 * h * f_nodes[..., k]
        + h01 * chi_nodes[..., k + 1]
        + h11 * h * f_nodes[..., k + 1]
    )


def transverse(chi: torch.Tensor, Ok: torch.Tensor) -> torch.Tensor:
    r"""S_k(χ): sinh for open, sin for closed and χ (1 + Ω_k χ²/6) near flat, so Ω_k = 0 is smooth."""
    root = Ok.abs().clamp_min(1e-12).sqrt()
    x = root * chi
    open_ = torch.sinh(x) / root
    closed = torch.sin(x) / root
    flat = chi * (1 + Ok * chi**2 / 6)
    return torch.where(Ok > 1e-6, open_, torch.where(Ok < -1e-6, closed, flat))


def luminosity_distance(z, nodes, H0, Om, OL) -> torch.Tensor:
    r"""d_L = (1+z) (c/H_0) S_k(χ(z)) in Mpc.   z: (N,), H0, Om, OL: (...) or (..., 1) → (..., N)"""
    H0, Om, OL = (torch.as_tensor(p, dtype=z.dtype, device=z.device) for p in (H0, Om, OL))
    if Om.dim() == 0:
        H0, Om, OL = H0[None], Om[None], OL[None]
    chi = interpolate(nodes, *comoving_integral(nodes, Om, OL), z)  # (..., N)
    return (1 + z) * C_KMS / H0 * transverse(chi, 1 - Om - OL)


def distance_modulus(d_L: torch.Tensor) -> torch.Tensor:
    """μ = 5 log10(d_L / Mpc) + 25 (25 = 5 log10 of 1 Mpc / 10 pc)."""
    return 5 * torch.log10(d_L) + 25


def chi_squared(mu_model: torch.Tensor, mu: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    r"""χ² = Σ_i ((μ_i − μ^{model}_i) / σ_i)²: einsum over the supernova index i.   (..., N) → (...)"""
    r = (mu - mu_model) / sigma
    return torch.einsum("...i,...i->...", r, r)


def deceleration(Om: torch.Tensor, OL: torch.Tensor) -> torch.Tensor:
    r"""q_0 = −\ddot a a / \dot a^2 today = Ω_m/2 − Ω_Λ: negative means the expansion is speeding up."""
    return Om / 2 - OL


def valid(nodes: torch.Tensor, Om: torch.Tensor, OL: torch.Tensor) -> torch.Tensor:
    """True where E² > 0 at every redshift up to the last supernova (a model the light could have crossed)."""
    return (e_squared(nodes, Om, OL) > 0).all(dim=-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/cosmological_constant_fit/tensors.py
"""Build the concrete tensors: load the supernova table, the redshift grid and the grid of candidate universes.

data/pantheon_plus.csv holds 1466 Type Ia supernovae from the Pantheon+ compilation (Scolnic et al. 2022, Brout et
al. 2022): name, redshift z (CMB frame, corrected for peculiar velocities), distance modulus mu (SH0ES-calibrated
absolute magnitude) and its diagonal error sigma_mu. Only z > 0.01 is kept (slower galaxies are dominated by their
own motions), the Cepheid-calibrator hosts are dropped and a supernova seen by several surveys appears once (the
row with the smallest error). Everything is float64: χ² differences of a few units on top of χ² ≈ 1400 need it.
"""

from pathlib import Path

import torch

from . import equation

DATA = Path(__file__).parent / "data" / "pantheon_plus.csv"
DTYPE = torch.float64
COLUMNS = {  # accepted header spellings (case-insensitive) for each column the fit needs
    "z": ("z", "zhd", "zcmb", "redshift"),
    "mu": ("mu", "mu_sh0es", "distance_modulus", "dm"),
    "sigma": ("sigma_mu", "mu_err", "dmu", "sigma", "mu_sh0es_err_diag", "error"),
}


def load_distance_moduli(path: Path = DATA, device: torch.device | None = None, z_min: float = 0.0):
    """Read a supernova CSV and return z, μ, σ_μ as float64 tensors of shape (N,), sorted by redshift.

    Works for any comma-separated file with a header row that names a redshift, a distance-modulus and an
    error column (see COLUMNS for the accepted names, any order, extra columns ignored, lines starting with #
    skipped), so an uploaded table can replace the bundled Pantheon+ one. Rows with z <= z_min are dropped.
    """
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    header = [h.strip().lower() for h in lines[0].split(",")]
    index = {}
    for key, names in COLUMNS.items():
        found = [header.index(n) for n in names if n in header]
        if not found:
            raise ValueError(f"no {key} column in {path}: expected one of {names}, got {header}")
        index[key] = found[0]
    rows = [ln.split(",") for ln in lines[1:]]
    table = torch.tensor([[float(r[index[k]]) for k in ("z", "mu", "sigma")] for r in rows], dtype=DTYPE)
    table = table[table[:, 0] > z_min]
    table = table[table[:, 0].argsort()].to(device)  # (N, 3)
    return table[:, 0], table[:, 1], table[:, 2]


def redshift_nodes(z_max: float, n: int, device: torch.device | None = None) -> torch.Tensor:
    """Uniform grid 0 … z_max (n nodes) on which the distance integral is accumulated: (n,)."""
    return torch.linspace(0.0, z_max, n, dtype=DTYPE, device=device)


def nodes_for(z: torch.Tensor, n: int = 1000) -> torch.Tensor:
    """Grid that reaches just past the farthest supernova."""
    return redshift_nodes(1.01 * float(z.max()), n, z.device)


def parameter_grid(n_m: int, n_l: int, range_m, range_l, device: torch.device | None = None):
    """Candidate universes: Ω_m values (A,) and Ω_Λ values (B,); the grid itself is their outer product."""
    Om = torch.linspace(*range_m, n_m, dtype=DTYPE, device=device)
    OL = torch.linspace(*range_l, n_l, dtype=DTYPE, device=device)
    return Om, OL


def synthetic_moduli(z, sigma, H0, Om, OL, noise: bool = True, generator: torch.Generator | None = None):
    """Distance moduli a universe (H0, Ω_m, Ω_Λ) would produce at the redshifts z, plus Gaussian errors σ."""
    mu = equation.distance_modulus(equation.luminosity_distance(z, nodes_for(z), H0, Om, OL))
    if noise:
        mu = mu + sigma * torch.randn(len(z), dtype=z.dtype, generator=generator).to(z.device)
    return mu


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/cosmological_constant_fit/solve.py
r"""Two ways to fit: a batched χ² over a whole grid of universes, and gradient descent with autograd.

On the grid, H_0 enters only as a constant shift of every μ (μ = 5 log10(d_L H_0) + 25 − 5 log10 H_0), so its best
value is found analytically for every cell: the inverse-variance mean of the residuals. Autograd instead moves
(H_0, Ω_m, Ω_Λ) together downhill with Adam and polishes with L-BFGS; the Hessian of χ² at the minimum gives the
error bars (covariance = 2 H^{-1}).
"""

import math

import torch

from problems._common.progress import steps as progress_steps

from . import equation, tensors


def chi2_of(p: torch.Tensor, z, mu, sigma, nodes=None) -> torch.Tensor:
    """χ² of one universe p = (H0, Ω_m, Ω_Λ), differentiable in p."""
    nodes = tensors.nodes_for(z) if nodes is None else nodes
    d_L = equation.luminosity_distance(z, nodes, p[0], p[1], p[2])  # (N,)
    return equation.chi_squared(equation.distance_modulus(d_L), mu, sigma)


def chi2_grid(z, mu, sigma, Om, OL, chunk: int = 8):
    """χ² (A, B) of every universe on the Ω_m × Ω_Λ grid, minimised over H0, and that best H0 (A, B).

    Rows of the grid are processed `chunk` at a time to keep the (rows, B, N) residual tensor small.
    """
    nodes = tensors.nodes_for(z)
    w = sigma**-2  # (N,) inverse variances
    chi2_rows, h0_rows = [], []
    for start in range(0, len(Om), chunk):
        om = Om[start : start + chunk, None, None]  # (a, 1, 1)
        ol = OL[None, :, None]  # (1, B, 1)
        d1 = equation.luminosity_distance(z, nodes, torch.ones_like(om), om, ol)  # (a, B, N), H0 = 1
        r = mu - equation.distance_modulus(d1)  # residuals before the H0 shift
        shift = torch.einsum("abi,i->ab", r, w) / w.sum()  # best −5 log10 H0, the weighted mean residual
        chi2 = torch.einsum("abi,abi,i->ab", r - shift[..., None], r - shift[..., None], w)
        ok = equation.valid(nodes, om, ol) & (d1 > 0).all(-1)  # (a, B); closed models can wrap past the antipode
        chi2_rows.append(torch.where(ok, chi2, torch.full_like(chi2, math.inf)))
        h0_rows.append(10 ** (-shift / 5))
    return torch.cat(chi2_rows), torch.cat(h0_rows)


def fit(z, mu, sigma, start=(70.0, 1.0, 0.0), steps: int = 400, lr: float = 0.02, flat: bool = False) -> dict:
    """Adam from `start` (default: the matter-only Einstein–de Sitter universe), then L-BFGS to the minimum.

    The optimiser works on (H0/100, Ω_m, Ω_Λ) so all three move on the same scale. With flat=True, Ω_Λ = 1 − Ω_m.
    Returns the path (S, 3), χ² along it (S,), the best parameters (3,) and their 1σ errors (3,).
    """
    nodes = tensors.nodes_for(z)
    q = torch.tensor([start[0] / 100, start[1], start[2]], dtype=z.dtype, device=z.device, requires_grad=True)

    def params(q):
        OL = 1 - q[1] if flat else q[2]
        return torch.stack([100 * q[0], q[1], OL])

    def loss(q):
        return chi2_of(params(q), z, mu, sigma, nodes)

    opt = torch.optim.Adam([q], lr=lr)
    path, chi2 = [], []
    for _ in progress_steps(steps, "Adam"):
        opt.zero_grad()
        value = loss(q)
        path.append(params(q).detach().clone())
        chi2.append(value.item())
        value.backward()
        opt.step()

    lbfgs = torch.optim.LBFGS([q], lr=1.0, max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad()
        value = loss(q)
        value.backward()
        return value

    lbfgs.step(closure)
    best = params(q).detach()
    path.append(best.clone())
    chi2.append(loss(q).item())

    free = [0, 1] if flat else [0, 1, 2]  # parameters the Hessian is taken over
    p_best = best[free].clone()

    def chi2_free(p):
        full = torch.stack([p[0], p[1], 1 - p[1]]) if flat else p
        return chi2_of(full, z, mu, sigma, nodes)

    hess = torch.autograd.functional.hessian(chi2_free, p_best)
    errors = torch.linalg.inv(hess).diagonal().mul(2).sqrt()  # covariance = 2 H⁻¹ for χ² = −2 ln L
    if flat:
        errors = torch.cat([errors, errors[1:]])  # σ(Ω_Λ) = σ(Ω_m) on the flat line
    return {"path": torch.stack(path), "chi2": torch.tensor(chi2), "best": best, "errors": errors}


def significance(chi2: torch.Tensor, Om: torch.Tensor, OL: torch.Tensor) -> dict:
    r"""How strongly the grid rejects Ω_Λ ≤ 0 and q_0 ≥ 0 (a decelerating universe).

    nσ = \sqrt{Δχ²} with Δχ² = min χ² inside the rejected region − global min (profile likelihood, one constraint).
    p_accel is the share of the likelihood e^{−χ²/2} (flat prior on the grid) with q_0 < 0.
    """
    om, ol = Om[:, None], OL[None, :]  # broadcast to the (A, B) grid
    q0 = equation.deceleration(om, ol).expand_as(chi2)
    best = chi2.min()
    i, j = divmod(int(chi2.argmin()), chi2.shape[1])
    no_lambda = chi2[(ol <= 0).expand_as(chi2)].min()
    decel = chi2[q0 >= 0].min()
    like = torch.exp(-(chi2 - best) / 2)  # 0 where χ² = inf
    return {
        "chi2_min": best.item(),
        "best_Om": Om[i].item(),
        "best_OL": OL[j].item(),
        "best_q0": q0[i, j].item(),
        "sigma_lambda": (no_lambda - best).clamp_min(0).sqrt().item(),
        "sigma_accel": (decel - best).clamp_min(0).sqrt().item(),
        "p_accel": (like[q0 < 0].sum() / like.sum()).item(),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/cosmological_constant_fit/plot.py
"""Animate the fit: the Hubble diagram relative to an empty universe, and the path through the Ω_m–Ω_Λ plane."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation, tensors

LEVELS = [2.30, 6.18, 11.83]  # Δχ² of the 1σ, 2σ, 3σ regions for two parameters


def binned(z, mu, sigma, n_bins: int = 16):
    """Inverse-variance mean of the residuals in log-spaced redshift bins (for the eye only, not for the fit)."""
    edges = np.geomspace(z.min(), z.max() * 1.001, n_bins + 1)
    k = np.digitize(z, edges) - 1
    w = sigma**-2
    sw = np.bincount(k, w, n_bins)
    keep = sw > 0
    zc = np.bincount(k, w * z, n_bins)[keep] / sw[keep]
    mc = np.bincount(k, w * mu, n_bins)[keep] / sw[keep]
    return zc, mc, sw[keep] ** -0.5


def animate(z, mu, sigma, fit, Om, OL, chi2, sig, path: Path) -> Path:
    zs = torch.logspace(np.log10(float(z.min())), np.log10(float(z.max())), 200, dtype=z.dtype)
    nodes = tensors.nodes_for(z)
    H0 = fit["best"][0]

    def mu_of(zz, h0, om, ol):
        return equation.distance_modulus(equation.luminosity_distance(zz, nodes, h0, om, ol)).numpy()

    empty_data = mu_of(z, H0, 0.0, 0.0)
    empty_curve = mu_of(zs, H0, 0.0, 0.0)
    res = (mu - torch.as_tensor(empty_data)).numpy()
    zn, sn = z.numpy(), sigma.numpy()
    zb, mb, eb = binned(zn, res, sn)
    path_p = fit["path"].numpy()  # (S, 3)
    curves = np.stack([mu_of(zs, *p) for p in fit["path"][frame_indices(len(path_p), 70)]])
    frames = frame_indices(len(path_p), 70)
    curve_of = dict(zip(frames, curves))
    eds = mu_of(zs, H0, 1.0, 0.0) - empty_curve
    best = mu_of(zs, *fit["best"]) - empty_curve

    fig, (ax, ax_c) = plt.subplots(1, 2, figsize=(9.2, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
    ax.scatter(zn, res, s=3, color="#d1d5db", lw=0, label=f"{len(zn)} SNe Ia (Pantheon+)")
    ax.errorbar(zb, mb, eb, fmt="o", ms=4, color="#374151", lw=1, capsize=0, label="binned mean")
    ax.axhline(0, color="#6b7280", lw=1, ls=":", label="empty universe (coasting)")
    ax.plot(zs.numpy(), eds, color=ACCENT[1], lw=1.5, label=r"matter only $\Omega_m = 1$ (decelerating)")
    ax.plot(zs.numpy(), best, color="#111827", lw=1.2, ls="--", label="best fit")
    current = ax.plot([], [], color=ACCENT[0], lw=2.2, label="Adam iterate")[0]
    ax.set_xscale("log")
    ax.set_ylim(-0.8, 0.6)
    ax.set_xlabel("redshift z")
    ax.set_ylabel(r"$\mu - \mu_\mathrm{empty}$ (mag)")
    ax.legend(fontsize=6.5, loc="lower left")
    ax.set_title("SNe farther (fainter) than in a coasting universe ⇒ acceleration", fontsize=8.5)

    om, ol = Om.numpy(), OL.numpy()
    dchi = (chi2 - chi2.min()).numpy().T  # (B, A) for contourf
    dchi = np.where(np.isfinite(dchi), dchi, 1e9)
    ax_c.contourf(om, ol, dchi, levels=[0, *LEVELS], colors=["#1d4ed8", "#60a5fa", "#bfdbfe"])
    ax_c.contour(om, ol, dchi, levels=LEVELS, colors="#1e40af", linewidths=0.6)
    invalid = ~np.isfinite((chi2 - chi2.min()).numpy().T)
    ax_c.contourf(om, ol, invalid.astype(float), levels=[0.5, 1.5], colors=["#e5e7eb"])
    ax_c.text(0.05, 1.45, "no Big Bang", fontsize=7, color="#6b7280")
    ax_c.plot(om, 1 - om, color="#374151", lw=1, ls="--")
    ax_c.text(0.13, 0.9, "flat", fontsize=7, color="#374151", rotation=-38)
    ax_c.plot(om, om / 2, color=ACCENT[3], lw=1)
    ax_c.text(0.75, 0.43, "q₀ = 0", fontsize=7, color=ACCENT[3], rotation=20)
    ax_c.text(0.75, 1.2, "accelerating", fontsize=7, color="#374151")
    ax_c.text(0.9, 0.25, "decelerating", fontsize=7, color="#374151")
    ax_c.axhline(0, color="#9ca3af", lw=0.8, ls=":")
    trail = ax_c.plot([], [], color=ACCENT[4], lw=1.2)[0]
    dot = ax_c.plot([], [], "o", color=ACCENT[4], ms=6, mec="white")[0]
    ax_c.set_xlim(om[0], om[-1])
    ax_c.set_ylim(ol[0], ol[-1])
    ax_c.set_xlabel(r"$\Omega_m$")
    ax_c.set_ylabel(r"$\Omega_\Lambda$")
    ax_c.set_title(f"1–3σ from the χ² grid · Ω_Λ ≤ 0 excluded at {sig['sigma_lambda']:.1f}σ", fontsize=8.5)
    label = ax_c.text(0.98, 0.02, "", transform=ax_c.transAxes, ha="right", va="bottom", fontsize=7.5)

    def update(step: int):
        p = path_p[step]
        current.set_data(zs.numpy(), curve_of[step] - empty_curve)
        trail.set_data(path_p[: step + 1, 1], path_p[: step + 1, 2])
        dot.set_data([p[1]], [p[2]])
        q0 = p[1] / 2 - p[2]
        label.set_text(
            f"step {step}: χ² = {fit['chi2'][step]:.0f}\n"
            f"H₀ ={p[0]:.1f}, Ω_m = {p[1]:.2f}, Ω_Λ = {p[2]:.2f}, q₀ = {q0:+.2f}"
        )

    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'geometry': 'free curvature',
    'start_Om': 1.0,
    'start_OL': 0.0,
    'steps': 400,
    'lr': 0.02,
    'grid': 121,
    'z_min': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.cosmological_constant_fit")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.cosmological_constant_fit.main")
problem_dir = Path("problems/cosmology/cosmological_constant_fit")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
